# Apollo on library material

First step of [ADR-0005](../docs/decisions/0005-baselines-and-prior-viability-on-real-library-material.md):
run the pretrained Apollo checkpoint on tracks from my own library and listen.

Apollo is a **baseline**, not a component. The question is how much of the damage it fixes on this
material, so that everything built later has a number and a reference to beat.

Two tracks: one MP3 (already lossy) and one WAV (clean, so Apollo is being asked to fix damage that
isn't there — a useful control).

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from grooveback import audio as ga
from grooveback.baselines import load_apollo, run_apollo, select_device

REPO = Path("..")  # notebook runs from notebooks/; Jupyter serves /files/ from the repo root
DATA = REPO / "data"
TRACKS = {
    "mp3": DATA / "AN-2 - Moonshine (Deep Boogie Version) [2003].mp3",
    # "wav": DATA / "Aerofunk - Nice One (Cpu Cant Hack It Mix) 258.wav",
}

print("device:", select_device("auto"))

## The source material

In [ ]:
sources = {}
for name, path in TRACKS.items():
    signal, sr = ga.load(path)
    sources[name] = (signal, sr)
    print(
        f"{name:4} {sr} Hz  {signal.shape[0]}ch  {signal.shape[1] / sr / 60:.1f} min  "
        f"{ga.loudness(signal, sr):6.1f} LUFS  peak {ga.peak_dbfs(signal):5.1f} dBFS"
    )
    print(f"     {path.name}")

## Pick an excerpt

Apollo runs at roughly **1.3x realtime on MPS**, so a full seven-minute track is about five and a half
minutes of compute. Excerpts keep this notebook interactive.

Set `EXCERPT_SECONDS = None` to process the whole track, and start it before making coffee.

In [ ]:
START_SECONDS = 0
# EXCERPT_SECONDS = 30.0
EXCERPT_SECONDS = None

excerpts = {}
for name, (signal, sr) in sources.items():
    if EXCERPT_SECONDS is None:
        excerpts[name] = (signal, sr)
    else:
        a = int(START_SECONDS * sr)
        b = a + int(EXCERPT_SECONDS * sr)
        excerpts[name] = (signal[:, a:b], sr)
    print(f"{name:4} {excerpts[name][0].shape[1] / sr:.1f}s")

## Run Apollo

The checkpoint downloads from Hugging Face on first use (~66 MB) and is cached after that.

Chunking is on by default. Apollo will happily try a whole track in one pass and exhaust 16 GB of
unified memory doing it.

In [ ]:
model = load_apollo()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

In [ ]:
restored = {}
for name, (signal, sr) in excerpts.items():
    started = time.perf_counter()
    restored[name] = run_apollo(signal, sr, model=model)
    elapsed = time.perf_counter() - started
    print(
        f"{name:4} {elapsed:5.1f}s  ({signal.shape[1] / sr / elapsed:.2f}x realtime)  "
        f"{ga.loudness(restored[name], sr):6.1f} LUFS  "
        f"peak {ga.peak_dbfs(restored[name]):5.1f} dBFS"
    )

## Write the output

Full length, level-matched to −14 LUFS. These files are both what gets played below and what goes to
the monitoring setup, which is the judgement that actually counts.

In [ ]:
OUT_DIR = Path("artifacts/apollo")

outputs = {}
for name, (signal, sr) in excerpts.items():
    paths = {"before": OUT_DIR / f"{name}_before.wav", "after": OUT_DIR / f"{name}_after.wav"}
    ga.save(REPO / paths["before"], ga.normalize_loudness(signal, sr), sr)
    ga.save(REPO / paths["after"], ga.normalize_loudness(restored[name], sr), sr)
    outputs[name] = paths
    print(f"{name:4} {paths['before']}  {paths['after']}")

## Listen

Both sides are level-matched to −14 LUFS. Without that the louder one wins regardless of whether it
is better, which is how most informal audio comparisons go wrong.

These stream from disk over Jupyter's `/files/` endpoint rather than being embedded. Passing a numpy
array to `Audio` base64-encodes the whole thing into the page — 107 MB per player for a seven-minute
track, saved into the `.ipynb` on every autosave. By URL it is 249 bytes and the browser streams it,
so full-length playback costs nothing.

Headphones or monitors. Laptop speakers will show none of this.

In [ ]:
for name, paths in outputs.items():
    print(f"=== {name}: {TRACKS[name].name}")
    print("before")
    display(Audio(url=f"/files/{paths['before']}"))
    print("after")
    display(Audio(url=f"/files/{paths['after']}"))

## Spectrograms

Scaled −120 to 0 dBFS, matching what an analyser like Spek shows, so these plots can be trusted
against one. What to look for: Apollo targets codec damage, so the interesting region is the top
octave — whether it restores air above ~16 kHz, and whether it invents a shelf that was never there.

The difference plot is the honest one. Red is energy Apollo added, blue is energy it removed.

In [ ]:
def plot_pair(name, floor_db=-120):
    signal, sr = excerpts[name]
    before = ga.spectrogram_db(ga.normalize_loudness(signal, sr))
    after = ga.spectrogram_db(ga.normalize_loudness(restored[name], sr))
    extent = [0, signal.shape[1] / sr, 0, sr / 2000]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
    for ax, data, title in [(axes[0], before, "before"), (axes[1], after, "after")]:
        im = ax.imshow(data, origin="lower", aspect="auto", extent=extent,
                       vmin=floor_db, vmax=0, cmap="magma")
        ax.set_title(title)
        ax.set_xlabel("time (s)")
    axes[0].set_ylabel("kHz")
    fig.colorbar(im, ax=axes[1], label="dBFS")

    delta = after - before
    limit = float(np.percentile(np.abs(delta), 99))
    im = axes[2].imshow(delta, origin="lower", aspect="auto", extent=extent,
                        vmin=-limit, vmax=limit, cmap="RdBu_r")
    axes[2].set_title("difference (red = added)")
    axes[2].set_xlabel("time (s)")
    fig.colorbar(im, ax=axes[2], label="dB")
    fig.suptitle(f"{name}: {TRACKS[name].name}", fontsize=11)
    plt.show()


for name in excerpts:
    plot_pair(name)

## Band energy

A colour map is easy to read optimistically. These are the numbers.

Read the **relative** column, not the absolute one. How much energy a track carries above 16 kHz
depends mostly on its master, so absolute dBFS says more about the record than about Apollo. What
matters is the top octave's level relative to the band beneath it, against the same ratio in a real
undamaged master.

In [ ]:
BANDS = [(20, 1000), (1000, 8000), (8000, 16000), (16000, 17000), (17000, 19000), (19000, 22050)]

for name, (signal, sr) in excerpts.items():
    before = ga.normalize_loudness(signal, sr)
    after = ga.normalize_loudness(restored[name], sr)
    print(f"=== {name}   (level-matched, dBFS)")
    print(f"{'band':>15} {'before':>9} {'after':>9} {'delta':>8}")
    for lo, hi in BANDS:
        b = ga.band_energy_db(before, sr, lo, hi)
        a = ga.band_energy_db(after, sr, lo, hi)
        print(f"{lo / 1000:6.0f}-{hi / 1000:5.1f} kHz {b:+9.1f} {a:+9.1f} {a - b:+8.1f}")

    ref_b = ga.band_energy_db(before, sr, 8000, 16000)
    ref_a = ga.band_energy_db(after, sr, 8000, 16000)
    top_b = ga.band_energy_db(before, sr, 16000, 22050)
    top_a = ga.band_energy_db(after, sr, 16000, 22050)
    print(f"  16-22k relative to 8-16k: before {top_b - ref_b:+.1f} dB, after {top_a - ref_a:+.1f} dB")
    print("  (a real 44.1k master sits around -20 dB here)\n")

## Average spectrum

The spectrograms show where the change is; this shows how much. A flat line at 0 dB means Apollo did
nothing at that frequency.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
for name, (signal, sr) in excerpts.items():
    before = ga.spectrogram_db(ga.normalize_loudness(signal, sr)).mean(axis=1)
    after = ga.spectrogram_db(ga.normalize_loudness(restored[name], sr)).mean(axis=1)
    freqs = np.linspace(0, sr / 2000, before.size)
    ax.plot(freqs, after - before, label=name, linewidth=1.2)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("kHz")
ax.set_ylabel("change (dB)")
ax.set_title("Apollo's average spectral change, level-matched")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Notes

Record what you hear here, then it goes into ADR-0005's results.

- Does it beat doing nothing, level-matched?
- What does it do to hats and rides — restored, or smeared into a hiss?
- What does it do to the **clean WAV**, where there is no codec damage to fix? Anything other than
  "almost nothing" means it is applying a learned house style rather than undoing damage.
- Any pumping or seams at the 10-second chunk boundaries?